In [ ]:
from FCN_model import FCN
import copy
import json
import random

import numpy as np
import optuna
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset


In [ ]:


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
MAX_EPOCHS = 100
PATIENCE = 15


In [ ]:


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(42)
        torch.cuda.manual_seed_all(42)
        torch.backends.cudnn.deterministic = True


set_seed(SEED)



In [ ]:
FordA_train = np.load("FordA_processed/FordA_TRAIN.npy")
FordA_test = np.load("FordA_processed/FordA_TEST.npy")

In [ ]:
FordA_X_train = np.expand_dims(FordA_train[:, 1:], axis=1)
FordA_y_train = FordA_train[:, 0]
FordA_y_train = (FordA_y_train == 1).astype(np.int64)

In [ ]:
FordA_X_test = np.expand_dims(FordA_test[:, 1:], axis=1)
FordA_y_test = FordA_test[:, 0]
FordA_y_test = (FordA_y_test == 1).astype(np.int64)

In [ ]:
FordA_X_train = torch.tensor(FordA_X_train, dtype=torch.float32)
FordA_y_train = torch.tensor(FordA_y_train, dtype=torch.long)

In [ ]:
FordA_X_test = torch.tensor(FordA_X_test, dtype=torch.float32)
FordA_y_test = torch.tensor(FordA_y_test, dtype=torch.long)

In [ ]:

# ---------------------------------------------------
# 1. Prepare training/validation datasets
# ---------------------------------------------------

train_idx, val_idx = train_test_split(
    np.arange(len(FordA_y_train)),
    test_size=0.2,
    stratify=FordA_y_train.numpy(),
    random_state=SEED
)

train_ds = TensorDataset(FordA_X_train[train_idx], FordA_y_train[train_idx])
val_ds = TensorDataset(FordA_X_train[val_idx], FordA_y_train[val_idx])
dataset = TensorDataset(FordA_X_train, FordA_y_train)

train_loader = DataLoader(
    train_ds,
    batch_size=360, # calculated as FordA_X_train.shape[0] / 10, according to tensorflow version
    shuffle=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=360, # calculated as FordA_X_train.shape[0] / 10, according to tensorflow version
    shuffle=True
)


In [ ]:

# ---------------------------------------------------
# 2. Define training and validation functions
# ---------------------------------------------------

def train_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(y_batch)
        correct += (logits.argmax(1) == y_batch).sum().item()
        total += len(y_batch)

    return total_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        total_loss += loss.item() * len(y_batch)
        correct += (logits.argmax(1) == y_batch).sum().item()
        total += len(y_batch)

    return total_loss / total, correct / total


# ---------------------------------------------------
# 3. Define the Optuna objective
# ---------------------------------------------------

def objective(trial):
    set_seed(SEED)

    lr = trial.suggest_float(
        "learning_rate", 1e-5, 1e-2, log=True
    )

    batch_size = trial.suggest_categorical(
        "batch_size", [16, 32, 64, 128]
    )

    weight_decay = trial.suggest_categorical(
        "weight_decay",
        [0.0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2]
    )

    optimizer_name = trial.suggest_categorical(
        "optimizer", ["Adam", "AdamW"]
    )

    # train_loader, val_loader = make_loaders(batch_size)

    # Replace with your existing FCN constructor.
    origin_FCN = FCN(in_channels = 1, num_classes=2).to("cuda")
    

    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(
            origin_FCN.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )
    else:
        optimizer = torch.optim.AdamW(
            origin_FCN.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )

    criterion = nn.CrossEntropyLoss()

    best_accuracy = -1.0
    best_loss = float("inf")
    best_epoch = 0
    patience_counter = 0
    best_state = None

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": []
    }

    for epoch in range(MAX_EPOCHS):

        train_loss, train_acc = train_epoch(
            origin_FCN, train_loader, optimizer, criterion
        )

        val_loss, val_acc = validate(
            origin_FCN, val_loader, criterion
        )

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)

        # Accuracy is the primary selection metric.
        # Use loss as a tie-breaker.
        improved = (
            val_acc > best_accuracy
            or (
                val_acc == best_accuracy
                and val_loss < best_loss
            )
        )

        if improved:
            best_accuracy = val_acc
            best_loss = val_loss
            best_epoch = epoch + 1
            best_state = copy.deepcopy(origin_FCN.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        # Report best validation accuracy so far.
        trial.report(best_accuracy, step=epoch)

        if trial.should_prune():
            raise optuna.TrialPruned()

        if patience_counter >= PATIENCE:
            break

    trial.set_user_attr("best_epoch", best_epoch)
    trial.set_user_attr("best_val_loss", best_loss)

    # Save each completed trial's best checkpoint.
    torch.save(
        {
            "model_state_dict": best_state,
            "params": trial.params,
            "best_epoch": best_epoch,
            "best_val_accuracy": best_accuracy,
            "history": history,
        },
        f"fcn_trial_{trial.number}.pt"
    )

    return best_accuracy


# ---------------------------------------------------
# 4. Run optimisation
# ---------------------------------------------------

study = optuna.create_study(
    study_name="FordA_FCN",
    storage="sqlite:///fcn_optuna.db",
    load_if_exists=True,
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=10
    )
)

study.optimize(
    objective,
    n_trials=50,
    gc_after_trial=True
)

print("Best validation accuracy:", study.best_value)
print("Best parameters:", study.best_params)
print("Best epoch:", study.best_trial.user_attrs["best_epoch"])

with open("best_params.json", "w") as f:
    json.dump(study.best_params, f, indent=4)

# Load the best trial's saved model.
checkpoint = torch.load(
    f"fcn_trial_{study.best_trial.number}.pt",
    map_location=DEVICE,
    weights_only=True
)

best_model = origin_FCN().to(DEVICE)
best_model.load_state_dict(checkpoint["model_state_dict"])
best_model.eval()
